# Internal migration and job reallocation (v1)

Do regions where people move in and out more also have more job churn? Internal (within-UK) migration is a measure of **worker mobility across places**; job reallocation measures **job mobility across firms**. In the US both have fallen together over recent decades (Molloy, Smith & Wozniak 2011, 2017; Hyatt & Spletzer 2013), which is often read as one "declining fluidity" story. Here: is the UK the same?

**Data** (ITL2 **2021**, 41 regions):
- Job flows: `clean/itl2_2021/job_flows_itl2.csv` (LBD).
- Migration: `clean/itl2_2021/migration_itl2.csv` (built in `build/migration.ipynb`): **moves crossing the region boundary** (`inflow_itl2`, `outflow_itl2`; moves between local authorities inside a region are excluded). Years ending June **2012–2024**. Sources: ONS (England & Wales and Northern Ireland, from GP-registration-based estimates) and National Records of Scotland (Scottish regions).
- Population: mid-year estimates (`age_itl2.csv`), as the denominator.

**Three versions of gross migration**, from narrowest to widest exclusion of short moves:
1. **All cross-region moves:** every move crossing an ITL2 boundary (moves between councils inside a region are already excluded).
2. **Excluding moves between London's five regions** (e.g. Camden → Hackney).
3. **Longer-distance moves (main measure):** only moves crossing an **ITL1** boundary, i.e. between the nine English regions, Wales, Scotland and Northern Ireland. This drops all moves between neighbouring ITL2s in the same region (London ↔ London, Greater Manchester ↔ Lancashire, Eastern ↔ West Central Scotland), which removes most of the boundary effect described below. Built from the region-to-region matrix (`migration_flows_itl2.csv`). For Scottish regions it is total cross-region moves minus moves within Scotland (NRS gives flows to the rest of the UK only as one total).

**Measures** (pooled 2012–24 from counts, as % of population per year): **gross migration rate** = (inflow + outflow) ÷ population; inflow and outflow rates; **net migration rate** = (inflow − outflow) ÷ population. Reallocation is pooled over the same years.

**Things to keep in mind:**
- **Geography matters for gross migration.** A move counts only if it crosses an ITL2 boundary, so small, closely packed regions (London's five) record far more "migration" than large, self-contained ones (Highlands and Islands, Northern Ireland as one region). Results are therefore also shown for **England & Wales outside London**, where regions are similar in kind and the data come from one source.
- **Scotland and NI** come from different sources and geographies (NI is one region, so it only records moves to and from GB). Their low gross rates are partly mechanical.
- **Method change:** ONS changed its internal migration method from the year ending June 2023; robustness checks stop in 2022.
- Descriptive, not causal: migration responds to jobs, and jobs to workers.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy import stats

BASE  = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
DIR21 = BASE / "clean" / "itl2_2021"
FIG   = BASE / "analysis" / "demographics" / "charts"
sys.path.append(str(BASE / "analysis"))
from chart_setup import styles, alt, ECO, TXT, GREY, place_labels

SOURCE = ("Sources: ONS Longitudinal Business Database (Secure Access, SN 9501); ONS and National Records of Scotland internal migration; "
          "ONS, NRS and NISRA population estimates; authors' calculations.")
MAIN = (2012, 2024)
FLOWS = ["jc_entry", "jd_exit", "jc_incumbent_ext", "jd_incumbent_ext", "jc_incumbent_int", "jd_incumbent_int"]
LONDON = lambda d: d.itl2_code.str.startswith("TLI")
EW = lambda d: d.itl2_code.str[:3].isin([f"TL{c}" for c in "CDEFGHIJKL"])         # England & Wales (incl. London)

reg = pd.read_csv(DIR21 / "job_flows_itl2.csv")
jf = reg[reg.analysis_window & ~((reg.itl2_code == "TLN0") & (reg.year < 2011))].copy()
mig = pd.read_csv(DIR21 / "migration_itl2.csv")
pop = pd.read_csv(DIR21 / "age_itl2.csv")[["itl2_code", "year", "population", "median_age"]]
names = pd.read_csv(DIR21 / "geography_spine.csv").drop_duplicates("itl2_code").set_index("itl2_code").itl2_name
pwd = pd.read_csv(DIR21 / "pop_weighted_density_itl2.csv").query("year == 2021").set_index("itl2_code").pop_weighted_density

# moves by type, per region-year, from the region-to-region matrix (Scotland appears as one block S92000003 = ITL1 TLM)
fl = pd.read_csv(DIR21 / "migration_flows_itl2.csv").query("origin != destination")
itl1 = lambda s: s.str[:3].where(s.str.startswith("TL"), "TLM")
fl["cross_itl1"] = itl1(fl.origin) != itl1(fl.destination)
fl["lon_lon"] = fl.origin.str.startswith("TLI") & fl.destination.str.startswith("TLI")
fl["scot_scot"] = fl.origin.str.startswith("TLM") & fl.destination.str.startswith("TLM")
side = lambda mask, end: fl[mask].groupby([end, "year"]).moves.sum().rename_axis(["itl2_code", "year"])
mv = mig.set_index(["itl2_code", "year"])[["inflow_itl2", "outflow_itl2"]].join(pd.DataFrame({
    "in_long": side(fl.cross_itl1, "destination"), "out_long": side(fl.cross_itl1, "origin"),
    "in_lonlon": side(fl.lon_lon, "destination"), "out_lonlon": side(fl.lon_lon, "origin"),
    "in_scot": side(fl.scot_scot, "destination"), "out_scot": side(fl.scot_scot, "origin")})).fillna(0)
sc = mv.index.get_level_values(0).str.startswith("TLM")      # Scottish regions: rest-of-UK flows only as a total
mv.loc[sc, "in_long"] = mv.loc[sc, "inflow_itl2"] - mv.loc[sc, "in_scot"]
mv.loc[sc, "out_long"] = mv.loc[sc, "outflow_itl2"] - mv.loc[sc, "out_scot"]
mv["in_xlon"], mv["out_xlon"] = mv.inflow_itl2 - mv.in_lonlon, mv.outflow_itl2 - mv.out_lonlon
mig = mig.merge(mv[["in_long", "out_long", "in_xlon", "out_xlon"]].reset_index(), on=["itl2_code", "year"])

def frame(years):
    g = jf[jf.year.between(*years)].groupby("itl2_code")[FLOWS + ["total_jc", "total_jd", "emp_avg"]].sum()
    y = pd.DataFrame({"realloc": 100 * (g.total_jc + g.total_jd) / g.emp_avg, "net_job": 100 * (g.total_jc - g.total_jd) / g.emp_avg,
                      "entry_exit": 100 * (g.jc_entry + g.jd_exit) / g.emp_avg, "incumbent": 100 * g[FLOWS[2:]].sum(axis=1) / g.emp_avg})
    m = mig[mig.year.between(*years)].merge(pop, on=["itl2_code", "year"])
    s = m.groupby("itl2_code")[["inflow_itl2", "outflow_itl2", "in_long", "out_long", "in_xlon", "out_xlon", "population"]].sum()
    x = pd.DataFrame({"in_rate": 100 * s.inflow_itl2 / s.population, "out_rate": 100 * s.outflow_itl2 / s.population,
                      "gross_xlon": 100 * (s.in_xlon + s.out_xlon) / s.population,
                      "in_long": 100 * s.in_long / s.population, "out_long": 100 * s.out_long / s.population})
    x["gross"], x["net"] = x.in_rate + x.out_rate, x.in_rate - x.out_rate
    x["gross_long"] = x.in_long + x.out_long
    d = y.join(x, how="inner").join(np.log(pwd).rename("log_pwd")).join(m.groupby("itl2_code").median_age.mean().rename("median_age")).join(names).reset_index()
    d["group"] = np.where(LONDON(d), "London", np.where(EW(d), "England & Wales", "Scotland & Northern Ireland"))
    return d

df = frame(MAIN)
assert len(df) == 41 and df.notna().all().all()
print(df.groupby("group")[["gross", "gross_xlon", "gross_long", "net", "realloc"]].mean().round(2).to_string())
print(f"Share of cross-region moves that cross an ITL1 boundary: {100 * mig.in_long.sum() / mig.inflow_itl2.sum():.0f}%")

                             gross  gross_xlon  gross_long   net  realloc
group                                                                    
England & Wales               7.33        7.33        5.36  0.26    27.08
London                       13.90        7.28        7.28 -1.18    32.91
Scotland & Northern Ireland   3.76        3.76        1.56  0.13    24.70
Share of cross-region moves that cross an ITL1 boundary: 68%


## 1. Across regions: is more migration linked to more churn?

In [2]:
def partial(d, x, yv, controls):
    Z = np.column_stack([np.ones(len(d))] + [d[c].values for c in controls])
    res = lambda v: v - Z @ np.linalg.lstsq(Z, v, rcond=None)[0]
    return np.corrcoef(res(d[x].values), res(d[yv].values))[0, 1]

SAMPLES = {"All 41 regions": df, "Excl. London (36)": df[~LONDON(df)], "England & Wales excl. London (30)": df[EW(df) & ~LONDON(df)]}
rows = []
for sname, s in SAMPLES.items():
    for x, lab in [("gross_long", "Gross, longer-distance (cross-ITL1)"), ("gross_xlon", "Gross, excl. London↔London"), ("gross", "Gross, all cross-region"), ("in_long", "Inflow, longer-distance"), ("out_long", "Outflow, longer-distance"), ("net", "Net migration")]:
        rows.append({"sample": sname, "measure": lab, "r": np.corrcoef(s.realloc, s[x])[0, 1], "rho": stats.spearmanr(s.realloc, s[x])[0],
                     "partial | density": partial(s, x, "realloc", ["log_pwd"]),
                     "partial | density + net job growth": partial(s, x, "realloc", ["log_pwd", "net_job"]),
                     "r with entry/exit": np.corrcoef(s.entry_exit, s[x])[0, 1], "r with incumbent flows": np.corrcoef(s.incumbent, s[x])[0, 1]})
print("Reallocation vs migration rates, pooled 2012–24"); print(pd.DataFrame(rows).round(2).to_string(index=False))

rob = []
for lab, yrs in [("2012–24", MAIN), ("2012–22 (before ONS method change)", (2012, 2022)), ("2012–19 (pre-COVID)", (2012, 2019))]:
    d = frame(yrs); s = d[EW(d) & ~LONDON(d)]
    rob.append({"window": lab, "r all": np.corrcoef(d.realloc, d.gross_long)[0, 1], "r excl. London": np.corrcoef(d[~LONDON(d)].realloc, d[~LONDON(d)].gross_long)[0, 1],
                "r E&W excl. London": np.corrcoef(s.realloc, s.gross_long)[0, 1]})
print("\nRobustness (longer-distance gross migration):"); print(pd.DataFrame(rob).round(2).to_string(index=False))

Reallocation vs migration rates, pooled 2012–24
                           sample                             measure     r   rho  partial | density  partial | density + net job growth  r with entry/exit  r with incumbent flows
                   All 41 regions Gross, longer-distance (cross-ITL1)  0.64  0.56               0.65                                0.49               0.70                    0.53
                   All 41 regions          Gross, excl. London↔London  0.43  0.36               0.69                                0.59               0.45                    0.36
                   All 41 regions             Gross, all cross-region  0.80  0.57               0.66                                0.50               0.84                    0.68
                   All 41 regions             Inflow, longer-distance  0.51  0.39               0.64                                0.46               0.53                    0.44
                   All 41 regions            Outflow

## 2. Over time: did migration fall as churn fell?

UK-wide rates each year from summed counts: reallocation (41 regions; NI from 2011) and gross cross-region migration (all moves crossing an ITL2 boundary, counting each move once as an outflow and once as an inflow ÷ UK population). Then within regions: a panel with region and year fixed effects asks whether a region's churn is higher in years when its migration is higher than usual.

In [3]:
yrs = list(range(MAIN[0], MAIN[1] + 1))
ukj = jf[jf.year.isin(yrs)].groupby("year")[["total_jc", "total_jd", "emp_avg"]].sum()
ukm = mig[mig.year.isin(yrs)].merge(pop, on=["itl2_code", "year"]).groupby("year")[["inflow_itl2", "outflow_itl2", "in_long", "out_long", "population"]].sum()
uk = pd.DataFrame({"Job reallocation rate": 100 * (ukj.total_jc + ukj.total_jd) / ukj.emp_avg,
                   "Longer-distance migration rate": 100 * (ukm.in_long + ukm.out_long) / ukm.population,
                   "All cross-region migration rate": 100 * (ukm.inflow_itl2 + ukm.outflow_itl2) / ukm.population})
print(uk.round(2).T.to_string())
early, late = uk.loc[2012:2014].mean(), uk.loc[2017:2019].mean()
print(f"\n2012–14 → 2017–19 (pre-COVID, same method): reallocation {early.iloc[0]:.1f} → {late.iloc[0]:.1f}; "
      f"longer-distance migration {early.iloc[1]:.2f} → {late.iloc[1]:.2f}; all cross-region {early.iloc[2]:.2f} → {late.iloc[2]:.2f}")

pan = (jf[["itl2_code", "year", "reallocation_rate"]].merge(mig, on=["itl2_code", "year"]).merge(pop, on=["itl2_code", "year"]))
pan["gross"] = 100 * (pan.in_long + pan.out_long) / pan.population          # longer-distance
pan["net"] = 100 * (pan.inflow_itl2 - pan.outflow_itl2) / pan.population
def fe(d, x):
    X = np.column_stack([d[x].values, pd.get_dummies(d.itl2_code, drop_first=True, dtype=float), pd.get_dummies(d.year, drop_first=True, dtype=float), np.ones(len(d))])
    yv = d.reallocation_rate.values; b = np.linalg.lstsq(X, yv, rcond=None)[0]; e = yv - X @ b; A = np.linalg.pinv(X.T @ X)
    G = pd.factorize(d.itl2_code)[0]; S = np.zeros((X.shape[1],) * 2)
    for gg in np.unique(G):
        sg = (X[G == gg] * e[G == gg][:, None]).sum(0); S += np.outer(sg, sg)
    n, k = X.shape; ng = G.max() + 1; V = A @ S @ A * ng / (ng - 1) * (n - 1) / (n - k)
    return b[0], np.sqrt(V[0, 0]), n
print("\nPanel, region + year fixed effects (pp reallocation per 1 pp of migration rate; clustered SE):")
for x in ["gross", "net"]:
    for lab, d in [("2012–24", pan), ("2012–22", pan[pan.year <= 2022])]:
        b, se, n = fe(d, x); print(f"  {x:5} {lab}: {b:+.2f} (se {se:.2f}, t {b / se:+.1f}), N = {n}")

year                              2012   2013   2014   2015   2016   2017   2018   2019   2020   2021   2022   2023   2024
Job reallocation rate            30.84  30.47  27.57  28.48  28.91  27.34  29.52  27.76  26.41  25.21  27.51  30.68  28.86
Longer-distance migration rate    4.68   4.66   4.99   5.03   5.00   5.49   5.44   5.51   5.06   5.95   5.54   5.40   5.43
All cross-region migration rate   6.90   6.82   7.31   7.32   7.26   8.02   7.94   8.07   7.35   8.76   8.12   7.91   7.99

2012–14 → 2017–19 (pre-COVID, same method): reallocation 29.6 → 28.2; longer-distance migration 4.77 → 5.48; all cross-region 7.01 → 8.01

Panel, region + year fixed effects (pp reallocation per 1 pp of migration rate; clustered SE):
  gross 2012–24: -0.14 (se 0.42, t -0.3), N = 533
  gross 2012–22: -0.25 (se 0.48, t -0.5), N = 451
  net   2012–24: +0.63 (se 0.39, t +1.6), N = 533
  net   2012–22: +0.83 (se 0.46, t +1.8), N = 451


## 3. Charts

In [4]:
# (a) gross migration vs reallocation, coloured by area type
W, H = 540, 380
d = df.copy()
XS = alt.Scale(domain=[0, float(np.ceil(d.gross_long.max() + 1))], nice=False)
YS = alt.Scale(domain=[21, 39], zero=False, nice=False)
GORDER = ["London", "England & Wales", "Scotland & Northern Ireland"]; GCOLS = [ECO["red"], ECO["blue-light"], ECO["orange"]]
ew = d[EW(d) & ~LONDON(d)]
r_all, r_ew = np.corrcoef(d.realloc, d.gross_long)[0, 1], np.corrcoef(ew.realloc, ew.gross_long)[0, 1]
labs = place_labels(d, "gross_long", "realloc", XS, YS, W, H, max_labels=7, max_len=22, pad=2)
sc = alt.layer(
    alt.Chart(d).mark_circle(size=90, opacity=0.9, stroke="white", strokeWidth=0.8).encode(
        x=alt.X("gross_long:Q", scale=XS, title="Longer-distance migration, 2012–24 (moves in + out between English regions and nations, % of population a year)"),
        y=alt.Y("realloc:Q", scale=YS, title="Job reallocation rate, 2012–24 (%)"),
        color=alt.Color("group:N", scale=alt.Scale(domain=GORDER, range=GCOLS), legend=alt.Legend(title=None, orient="bottom")),
        tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("gross_long:Q", format=".1f", title="Longer-distance migration (%)"), alt.Tooltip("gross:Q", format=".1f", title="All cross-region (%)"),
                 alt.Tooltip("net:Q", format="+.2f", title="Net migration (%)"), alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
    *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=10, color=TXT)
        .encode(x=alt.X("gross_long:Q", scale=XS), y=alt.Y("realloc:Q", scale=YS), text="short:N") for i, r in labs.iterrows()],
).properties(title=alt.Title("Regions with more migration tend to churn more, but the link is weak outside London, Scotland and NI",
    subtitle=[f"ITL2 regions, 2012–24. Correlation {r_all:.2f} across all regions; {r_ew:.2f} for England & Wales outside London.",
              "Moves between ITL2s in the same English region or nation (e.g. within London) excluded. Scotland and NI data come from different sources.", SOURCE]))
styles.save(sc, str(FIG), "migration_reallocation_scatter", width=W, height=H)
sc.properties(width=W, height=H)

alt.LayerChart(...)

In [5]:
# (b) UK over time: reallocation vs gross migration, indexed 2012 = 100
uk2 = uk[["Job reallocation rate", "Longer-distance migration rate"]]
idx = (100 * uk2 / uk2.loc[2012]).reset_index(names="year").melt(id_vars="year", var_name="series", value_name="index")
raw = uk2.reset_index(names="year").melt(id_vars="year", var_name="series", value_name="rate")
idx = idx.merge(raw, on=["year", "series"])
SORD = ["Longer-distance migration rate", "Job reallocation rate"]; SCOL = [ECO["orange"], ECO["blue-dark"]]
end = idx[idx.year == idx.year.max()]
XT = alt.Scale(domain=[2012, 2024], nice=False)
YT = alt.Scale(domain=[75, 130], nice=False)
xe = lambda f, **k: alt.X(f, scale=XT, **k)
ye = lambda f, **k: alt.Y(f, scale=YT, **k)
tl = alt.layer(
    alt.Chart(pd.DataFrame({"x": [2022.5]})).mark_rule(strokeDash=[3, 3], color=GREY).encode(x=xe("x:Q")),
    alt.Chart(pd.DataFrame({"x": [2022.4], "y": [128], "t": ["ONS method change →"]})).mark_text(align="right", fontSize=10, color=TXT)
       .encode(x=xe("x:Q"), y=ye("y:Q"), text="t:N"),
    alt.Chart(pd.DataFrame({"v": [100.0]})).mark_rule(color=TXT, opacity=0.4).encode(y=ye("v:Q")),
    alt.Chart(idx).mark_line(strokeWidth=2.8, point=alt.OverlayMarkDef(size=40)).encode(
        x=xe("year:Q", title=None, axis=alt.Axis(format="d", values=list(range(2012, 2025, 2)))),
        y=ye("index:Q", title="Index, 2012 = 100"),
        color=alt.Color("series:N", scale=alt.Scale(domain=SORD, range=SCOL), legend=None),
        tooltip=["series:N", "year:Q", alt.Tooltip("rate:Q", format=".1f", title="Rate (%)"), alt.Tooltip("index:Q", format=".0f")]),
    alt.Chart(end).mark_text(align="left", dx=8, fontSize=11, fontWeight="bold").encode(
        x=xe("year:Q"), y=ye("index:Q"), text="series:N", color=alt.Color("series:N", scale=alt.Scale(domain=SORD, range=SCOL), legend=None)),
).properties(title=alt.Title("Job churn fell while people kept moving between regions",
    subtitle=[f"UK, index 2012 = 100. Job reallocation rate (2012: {uk.iloc[0, 0]:.1f}%) and longer-distance migration rate, moves in + out between English regions and nations (2012: {uk.iloc[0, 1]:.1f}% of population).",
              "Migration years end in June. ONS changed its migration method from 2023; 2020–21 affected by COVID.", SOURCE]))
styles.save(tl, str(FIG), "migration_reallocation_uk_trend", width=520, height=330)
tl.properties(width=520, height=330)

alt.LayerChart(...)

In [6]:
# (c) by region: longer-distance inflow and outflow (butterfly) next to the reallocation rate, sorted by reallocation
d = df.assign(net_long=df.in_long - df.out_long)
d["short"] = d.itl2_name.str.replace(" - ", " ", regex=False).str.replace(" and ", " & ", regex=False)
order = list(d.sort_values("realloc", ascending=False).short)
bf = pd.concat([d.assign(flow="Moving out", rate=-d.out_long), d.assign(flow="Moving in", rate=d.in_long)])
yax = alt.Y("short:N", sort=order, title=None, axis=alt.Axis(labelFontSize=10, labelLimit=260, ticks=False, domain=False))
FCOL = alt.Color("flow:N", scale=alt.Scale(domain=["Moving out", "Moving in"], range=[ECO["orange"], ECO["turquoise"]]),
                 legend=alt.Legend(title=None, orient="top"))
lim = float(np.ceil(max(d.in_long.max(), d.out_long.max())))
butterfly = alt.layer(
    alt.Chart(bf).mark_bar(height=9).encode(
        y=yax, x=alt.X("rate:Q", scale=alt.Scale(domain=[-lim, lim]), title="Longer-distance migration, % of population a year",
                       axis=alt.Axis(labelExpr="abs(datum.value)")),
        color=FCOL, tooltip=["itl2_name:N", "flow:N", alt.Tooltip("rate:Q", format=".2f", title="% of population (out shown negative)")]),
    alt.Chart(d).mark_tick(color=TXT, thickness=2, size=11).encode(y=yax, x=alt.X("net_long:Q"),
        tooltip=["itl2_name:N", alt.Tooltip("net_long:Q", format="+.2f", title="Net, longer-distance (%)")]),
    alt.Chart(pd.DataFrame({"v": [0.0]})).mark_rule(color=TXT, opacity=0.6).encode(x="v:Q"),
).properties(width=300, height=620, title=alt.Title("People moving out and in", subtitle="Black tick = net (in minus out)", fontSize=13, anchor="start"))

colr = alt.Color("group:N", scale=alt.Scale(domain=["London", "England & Wales", "Scotland & Northern Ireland"],
                                            range=[ECO["red"], ECO["blue-dark"], ECO["blue-light"]]), legend=alt.Legend(title=None, orient="top"))
realloc_bar = alt.layer(
    alt.Chart(d).mark_bar(height=9).encode(y=alt.Y("short:N", sort=order, axis=None), x=alt.X("realloc:Q", title="Job reallocation rate, %", scale=alt.Scale(domain=[0, 40])),
                                           color=colr, tooltip=["itl2_name:N", alt.Tooltip("realloc:Q", format=".1f")]),
    alt.Chart(d).mark_text(align="left", dx=3, fontSize=9, color=TXT).encode(y=alt.Y("short:N", sort=order, axis=None), x="realloc:Q", text=alt.Text("realloc:Q", format=".1f")),
).properties(width=170, height=620, title=alt.Title("Job reallocation", fontSize=13, anchor="start"))

byreg = alt.hconcat(butterfly, realloc_bar, spacing=10).resolve_scale(color="independent").properties(title=alt.Title(
    "High-churn regions see more people leaving, and usually more arriving too",
    subtitle=["ITL2 regions sorted by job reallocation rate, 2012–24. Migration: moves in and out between English regions and nations",
              "(moves within the same English region or nation excluded), % of population a year.", SOURCE]))
styles.save(byreg, str(FIG), "migration_reallocation_by_region", width=300, height=620)
print("r(realloc, in_long) =", round(np.corrcoef(d.realloc, d.in_long)[0, 1], 2), "| r(realloc, out_long) =", round(np.corrcoef(d.realloc, d.out_long)[0, 1], 2))
byreg

r(realloc, in_long) = 0.51 | r(realloc, out_long) = 0.7


alt.HConcatChart(...)

In [7]:
# (d) two scatters: moving out and moving in (longer-distance) vs reallocation, shared axes
d = df.copy()
XS2 = alt.Scale(domain=[0, 5], nice=False)
YS2 = alt.Scale(domain=[21, 39], zero=False, nice=False)
W2, H2 = 330, 330
GORDER = ["London", "England & Wales", "Scotland & Northern Ireland"]; GCOLS = [ECO["red"], ECO["blue-light"], ECO["orange"]]
def io_panel(col, title, first):
    r_all = np.corrcoef(d.realloc, d[col])[0, 1]
    ew = d[EW(d) & ~LONDON(d)]; r_ew = np.corrcoef(ew.realloc, ew[col])[0, 1]
    b = np.polyfit(d[col], d.realloc, 1); fx = np.array([d[col].min(), d[col].max()])
    labs = place_labels(d, col, "realloc", XS2, YS2, W2, H2, max_labels=4, max_len=20, pad=2)
    return alt.layer(
        alt.Chart(pd.DataFrame({"x": fx, "y": np.polyval(b, fx)})).mark_line(color=GREY, strokeWidth=2, strokeDash=[6, 4])
           .encode(x=alt.X("x:Q", scale=XS2), y=alt.Y("y:Q", scale=YS2)),
        alt.Chart(d).mark_circle(size=70, opacity=0.9, stroke="white", strokeWidth=0.8).encode(
            x=alt.X(f"{col}:Q", scale=XS2, title="% of population a year", axis=alt.Axis(values=[0, 1, 2, 3, 4, 5])),
            y=alt.Y("realloc:Q", scale=YS2, title="Job reallocation rate, 2012–24 (%)" if first else None, axis=alt.Axis(labels=first, ticks=first)),
            color=alt.Color("group:N", scale=alt.Scale(domain=GORDER, range=GCOLS), legend=alt.Legend(title=None, orient="bottom") if first else None),
            tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip(f"{col}:Q", format=".2f", title=title),
                     alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
        *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=9, color=TXT)
            .encode(x=alt.X(f"{col}:Q", scale=XS2), y=alt.Y("realloc:Q", scale=YS2), text="short:N") for i, r in labs.iterrows()],
    ).properties(width=W2, height=H2, title=alt.Title(title, subtitle=f"r = {r_all:.2f} all regions; {r_ew:.2f} England & Wales excl. London",
                                                       fontSize=13, subtitleFontSize=11, anchor="start"))

io = alt.hconcat(io_panel("out_long", "People moving out", True), io_panel("in_long", "People moving in", False), spacing=20).resolve_scale(color="shared").properties(
    title=alt.Title("Job churn is linked more closely to people leaving a region than to people arriving",
        subtitle=["ITL2 regions, 2012–24. Longer-distance moves only (between English regions and nations; moves within the same English region or nation excluded).",
                  "Dashed line = fit across all regions.", SOURCE]))
styles.save(io, str(FIG), "migration_reallocation_in_out", width=W2, height=H2)
io

alt.HConcatChart(...)

In [8]:
# (e) net longer-distance migration (in - out) vs reallocation
d = df.assign(net_long=df.in_long - df.out_long)
# London as one area: sum its five regions' counts. Longer-distance moves already exclude London<->London moves,
# so the summed in/out flows are London's exchanges with the rest of the UK.
mL = mig[mig.year.between(*MAIN) & LONDON(mig)].merge(pop, on=["itl2_code", "year"])[["in_long", "out_long", "population"]].sum()
jL = jf[jf.year.between(*MAIN) & LONDON(jf)][["total_jc", "total_jd", "emp_avg"]].sum()
lon_row = pd.DataFrame([{"itl2_code": "TLI", "itl2_name": "London", "group": "London",
                         "in_long": 100 * mL.in_long / mL.population, "out_long": 100 * mL.out_long / mL.population,
                         "realloc": 100 * (jL.total_jc + jL.total_jd) / jL.emp_avg}])
lon_row["net_long"] = lon_row.in_long - lon_row.out_long
d = pd.concat([d[~LONDON(d)], lon_row], ignore_index=True)
print(f"London as one area: net {lon_row.net_long.iloc[0]:+.2f}% a year, reallocation {lon_row.realloc.iloc[0]:.1f}%")
W3, H3 = 540, 380
XS3 = alt.Scale(domain=[-1.5, 1.5], nice=False); YS3 = alt.Scale(domain=[21, 39], zero=False, nice=False)
GORDER = ["London", "England & Wales", "Scotland & Northern Ireland"]; GCOLS = [ECO["red"], ECO["blue-light"], ECO["orange"]]
nl = d[d.group != "London"]
r_all, r_nl = np.corrcoef(d.realloc, d.net_long)[0, 1], np.corrcoef(nl.realloc, nl.net_long)[0, 1]
print(f"net (longer-distance) vs reallocation: r = {r_all:.2f} all, {r_nl:.2f} excl. London; "
      f"regions gaining people: {(d.net_long > 0).sum()}, losing: {(d.net_long < 0).sum()}")
print(d.sort_values("net_long")[["itl2_name", "net_long", "realloc"]].round(2).iloc[[0, 1, 2, 3, 4, 5, -4, -3, -2, -1]].to_string(index=False))
labs = place_labels(d, "net_long", "realloc", XS3, YS3, W3, H3, max_labels=8, max_len=22, pad=2)
uk = 100 * jf[jf.year.between(*MAIN)][["total_jc", "total_jd"]].sum().sum() / jf[jf.year.between(*MAIN)].emp_avg.sum()
corner = pd.DataFrame({"x": [-1.47, 1.47], "y": [38.6, 38.6], "t": ["← Losing people", "Gaining people →"], "a": ["left", "right"]})
net_ch = alt.layer(
    alt.Chart(pd.DataFrame({"v": [0.0]})).mark_rule(color=TXT, opacity=0.6).encode(x=alt.X("v:Q", scale=XS3)),
    alt.Chart(pd.DataFrame({"v": [uk]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YS3)),
    alt.Chart(pd.DataFrame({"x": [1.5], "y": [uk]})).mark_text(align="right", baseline="bottom", dx=-3, dy=-3, fontSize=10, color=TXT)
       .encode(x=alt.X("x:Q", scale=XS3), y=alt.Y("y:Q", scale=YS3), text=alt.value(f"UK {uk:.1f}%")),
    *[alt.Chart(corner.iloc[[i]]).mark_text(align=r["a"], fontSize=11, fontWeight="bold", color=TXT, opacity=0.7)
        .encode(x=alt.X("x:Q", scale=XS3), y=alt.Y("y:Q", scale=YS3), text="t:N") for i, r in corner.iterrows()],
    alt.Chart(d).mark_circle(size=90, opacity=0.9, stroke="white", strokeWidth=0.8).encode(
        x=alt.X("net_long:Q", scale=XS3, title="Net migration (moves in minus moves out, % of population a year)",
                axis=alt.Axis(values=[-1.5, -1, -0.5, 0, 0.5, 1, 1.5], labelExpr="datum.value > 0 ? '+' + datum.label + '%' : datum.label + '%'")),
        y=alt.Y("realloc:Q", scale=YS3, title="Job reallocation rate", axis=alt.Axis(labelExpr="datum.label + '%'")),
        color=alt.Color("group:N", scale=alt.Scale(domain=GORDER, range=GCOLS), legend=alt.Legend(title=None, orient="bottom")),
        tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("in_long:Q", format=".2f", title="Moving in (%)"),
                 alt.Tooltip("out_long:Q", format=".2f", title="Moving out (%)"), alt.Tooltip("net_long:Q", format="+.2f", title="Net (%)"),
                 alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
    *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=10, color=TXT)
        .encode(x=alt.X("net_long:Q", scale=XS3), y=alt.Y("realloc:Q", scale=YS3), text="short:N") for i, r in labs.iterrows()],
).properties(title=alt.Title("London loses people to the rest of the UK; elsewhere, net migration says little about churn",
    subtitle=[f"ITL2 regions, 2012–24. Longer-distance moves only (between English regions and nations). London shown as one area. Correlation {r_all:.2f}; {r_nl:.2f} excluding London.",
              "Net figures exclude international migration, which adds to population in London especially.", SOURCE]))
styles.save(net_ch, str(FIG), "migration_reallocation_net", width=W3, height=H3)
net_ch.properties(width=W3, height=H3)

London as one area: net -1.21% a year, reallocation 34.4%
net (longer-distance) vs reallocation: r = -0.51 all, -0.24 excl. London; regions gaining people: 27, losing: 10
                                     itl2_name  net_long  realloc
                                        London     -1.21    34.44
                   Hampshire and Isle of Wight     -0.12    27.22
                                West Yorkshire     -0.09    27.46
Herefordshire, Worcestershire and Warwickshire     -0.06    27.98
                               South Yorkshire     -0.05    28.16
                              Northern Ireland     -0.04    22.53
                           Dorset and Somerset      0.58    25.06
                                         Essex      0.60    28.02
                                         Devon      0.69    24.22
                  Cornwall and Isles of Scilly      0.96    26.68


alt.LayerChart(...)

In [9]:
# Net migration vs reallocation, London excluded, coloured by nation (2012-24, longer-distance moves)
from chart_setup import place_labels as place_labels_g
NY = (2018, 2024)
df18 = frame(NY)
dn = df18[~LONDON(df18)].assign(net_long=lambda x: x.in_long - x.out_long)
dn["nation"] = dn.itl2_code.str[:3].map({"TLL": "Wales", "TLM": "Scotland", "TLN": "Northern Ireland"}).fillna("England")
dn["itl2_name"] = dn.itl2_name.replace({"Berkshire, Buckinghamshire and Oxfordshire": "Berks, Bucks & Oxon", "Bedfordshire and Hertfordshire": "Beds & Herts",
    "Gloucestershire, Wiltshire and Bath/Bristol area": "Glos, Wilts & Bristol", "Leicestershire, Rutland and Northamptonshire": "Leics & Northants",
    "Herefordshire, Worcestershire and Warwickshire": "Herefs, Worcs & Warks", "Surrey, East and West Sussex": "Surrey & Sussex",
    "East Yorkshire and Northern Lincolnshire": "E Yorks & N Lincs", "Northumberland, and Tyne and Wear": "Tyneside & Northumb.",
    "Derbyshire and Nottinghamshire": "Derbys & Notts", "Shropshire and Staffordshire": "Shrops & Staffs", "Cornwall and Isles of Scilly": "Cornwall"})
NAT = ["England", "Wales", "Scotland", "Northern Ireland"]; NCOL = [ECO["blue-light"], ECO["turquoise"], ECO["blue-dark"], ECO["yellow"]]
r_n = np.corrcoef(dn.net_long, dn.realloc)[0, 1]
j = jf[jf.year.between(*NY) & ~LONDON(jf)]; avg_nl = 100 * j[["total_jc", "total_jd"]].sum().sum() / j.emp_avg.sum()
print(f"2018–24, excl. London (36 regions): r = {r_n:.2f}; average reallocation {avg_nl:.1f}%")
W5, H5 = 560, 400
XN = alt.Scale(domain=[float(np.floor(dn.net_long.min() * 5 - 0.5) / 5), float(np.ceil(dn.net_long.max() * 5 + 0.5) / 5)], nice=False)
YN = alt.Scale(domain=[float(np.floor(dn.realloc.min() - 0.5)), float(np.ceil(dn.realloc.max() + 0.5))], zero=False, nice=False)
b = np.polyfit(dn.net_long, dn.realloc, 1); fx = np.array([dn.net_long.min(), dn.net_long.max()])
labs = place_labels_g(dn.assign(group="x"), "net_long", "realloc", XN, YN, W5, H5, max_labels=14, max_len=22, pad=2)
net_nl = alt.layer(
    alt.Chart(pd.DataFrame({"v": [0.0]})).mark_rule(color=TXT, opacity=0.5).encode(x=alt.X("v:Q", scale=XN)),
    alt.Chart(pd.DataFrame({"v": [avg_nl]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YN)),
    alt.Chart(pd.DataFrame({"x": fx, "y": np.polyval(b, fx)})).mark_line(color=GREY, strokeWidth=1.8, opacity=0.7)
       .encode(x=alt.X("x:Q", scale=XN), y=alt.Y("y:Q", scale=YN)),
    alt.Chart(dn).mark_circle(size=80, opacity=0.95, stroke="white", strokeWidth=0.6).encode(
        x=alt.X("net_long:Q", scale=XN, title="Net migration (moves in minus moves out, % of population a year)",
                axis=alt.Axis(values=[round(v, 1) for v in np.arange(-1.0, 2.01, 0.2)], labelExpr="(datum.value > 0 ? '+' : '') + format(datum.value, '.1f') + '%'")),
        y=alt.Y("realloc:Q", scale=YN, title="Job reallocation rate", axis=alt.Axis(labelExpr="datum.label + '%'")),
        color=alt.Color("nation:N", scale=alt.Scale(domain=NAT, range=NCOL), legend=alt.Legend(title=None, orient="bottom")),
        tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("in_long:Q", format=".2f", title="Moving in (%)"),
                 alt.Tooltip("out_long:Q", format=".2f", title="Moving out (%)"), alt.Tooltip("net_long:Q", format="+.2f", title="Net (%)"),
                 alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
    *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=9, color=TXT)
        .encode(x=alt.X("net_long:Q", scale=XN), y=alt.Y("realloc:Q", scale=YN), text="short:N") for i, r in labs.iterrows()],
).properties(title=alt.Title("Outside London, regions gaining people churn no more than those losing them",
    subtitle=[f"ITL2 regions outside London, 2018–24 (job reallocation and migration both pooled). Longer-distance moves only (between English regions and nations). Correlation {r_n:.2f}. Grey line = fit.",
              f"Dotted line = average job reallocation outside London ({avg_nl:.1f}%). Net figures exclude international migration.", SOURCE]))
styles.save(net_nl, str(FIG), "migration_reallocation_net_excl_london", width=W5, height=H5)
net_nl.properties(width=W5, height=H5)

2018–24, excl. London (36 regions): r = -0.29; average reallocation 26.7%


alt.LayerChart(...)

In [10]:
# Two panels: first and last five years of the migration data (2012-16 vs 2020-24), London excluded
NP = {"2012–16": (2012, 2016), "2020–24": (2020, 2024)}
def nl_frame(yrs):
    d_ = frame(yrs); d_ = d_[~LONDON(d_)].assign(net_long=lambda x: x.in_long - x.out_long)
    d_["nation"] = d_.itl2_code.str[:3].map({"TLL": "Wales", "TLM": "Scotland", "TLN": "Northern Ireland"}).fillna("England")
    d_["itl2_name"] = d_.itl2_name.replace(dict(zip(df.itl2_name, df.itl2_name)) | {k: v for k, v in zip(
        ["Berkshire, Buckinghamshire and Oxfordshire", "Bedfordshire and Hertfordshire", "Gloucestershire, Wiltshire and Bath/Bristol area",
         "Leicestershire, Rutland and Northamptonshire", "Herefordshire, Worcestershire and Warwickshire", "Surrey, East and West Sussex",
         "East Yorkshire and Northern Lincolnshire", "Northumberland, and Tyne and Wear", "Derbyshire and Nottinghamshire",
         "Shropshire and Staffordshire", "Cornwall and Isles of Scilly"],
        ["Berks, Bucks & Oxon", "Beds & Herts", "Glos, Wilts & Bristol", "Leics & Northants", "Herefs, Worcs & Warks", "Surrey & Sussex",
         "E Yorks & N Lincs", "Tyneside & Northumb.", "Derbys & Notts", "Shrops & Staffs", "Cornwall"])})
    return d_
pdn = {lab: nl_frame(y) for lab, y in NP.items()}
allp = pd.concat(pdn.values())
XP = alt.Scale(domain=[float(np.floor(allp.net_long.min() * 5 - 0.5) / 5), float(np.ceil(allp.net_long.max() * 5 + 0.5) / 5)], nice=False)
YP = alt.Scale(domain=[float(np.floor(allp.realloc.min() - 0.5)), float(np.ceil(allp.realloc.max() + 0.5))], zero=False, nice=False)
WP, HP_ = 350, 360
def net_panel(lab, first):
    d_ = pdn[lab]; yrs = NP[lab]
    jj = jf[jf.year.between(*yrs) & ~LONDON(jf)]; av = 100 * jj[["total_jc", "total_jd"]].sum().sum() / jj.emp_avg.sum()
    r_ = np.corrcoef(d_.net_long, d_.realloc)[0, 1]; bb = np.polyfit(d_.net_long, d_.realloc, 1); fx_ = np.array([d_.net_long.min(), d_.net_long.max()])
    labs_ = place_labels_g(d_.assign(group="x"), "net_long", "realloc", XP, YP, WP, HP_, max_labels=11, max_len=22, pad=1)
    ax_y = alt.Axis(title="Job reallocation rate" if first else None, labels=first, ticks=first, labelFlush=True, labelExpr="datum.label + '%'")
    print(f"{lab}: r = {r_:.2f}, average reallocation {av:.1f}%, mean net {d_.net_long.mean():+.2f}%")
    return alt.layer(
        alt.Chart(pd.DataFrame({"v": [0.0]})).mark_rule(color=TXT, opacity=0.5).encode(x=alt.X("v:Q", scale=XP)),
        alt.Chart(pd.DataFrame({"v": [av]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YP, axis=ax_y)),
        alt.Chart(pd.DataFrame({"x": fx_, "y": np.polyval(bb, fx_)})).mark_line(color=GREY, strokeWidth=1.8, opacity=0.7)
           .encode(x=alt.X("x:Q", scale=XP), y=alt.Y("y:Q", scale=YP)),
        alt.Chart(d_).mark_circle(size=70, opacity=0.95, stroke="white", strokeWidth=0.6).encode(
            x=alt.X("net_long:Q", scale=XP, title="Net migration, % of population a year",
                    axis=alt.Axis(values=[round(v, 1) for v in np.arange(-0.8, 2.01, 0.4)], labelExpr="(datum.value > 0 ? '+' : '') + format(datum.value, '.1f') + '%'")),
            y=alt.Y("realloc:Q", scale=YP, axis=ax_y),
            color=alt.Color("nation:N", scale=alt.Scale(domain=NAT, range=NCOL), legend=alt.Legend(title=None, orient="bottom") if first else None),
            tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("net_long:Q", format="+.2f", title="Net (%)"),
                     alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
        *[alt.Chart(labs_.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=9, color=TXT)
            .encode(x=alt.X("net_long:Q", scale=XP), y=alt.Y("realloc:Q", scale=YP), text="short:N") for i, r in labs_.iterrows()],
    ).properties(width=WP, height=HP_, title=alt.Title(lab, subtitle=f"r = {r_:.2f}", anchor="start", fontSize=13, subtitleFontSize=11))

netp = alt.hconcat(*[net_panel(lab, i == 0) for i, lab in enumerate(NP)], spacing=14).resolve_scale(color="shared").properties(
    title=alt.Title("Outside London, net migration says little about job churn in either period",
    subtitle=["ITL2 regions outside London: net longer-distance migration (between English regions and nations) against job reallocation, pooled over each period.",
              "Same axes in both panels. Grey line = fit; dotted line = average reallocation outside London. 2020–24 includes COVID and ONS's 2023 migration method change.", SOURCE]))
styles.save(netp, str(FIG), "migration_reallocation_net_excl_london_periods", width=WP, height=HP_)
netp

2012–16: r = -0.11, average reallocation 28.0%, mean net +0.14%
2020–24: r = -0.21, average reallocation 26.5%, mean net +0.28%


alt.HConcatChart(...)

## 4. What this shows (and doesn't)

*(Numbers refer to the printed output above.)*

- **Main measure: longer-distance moves** (crossing English region / nation boundaries; 68% of all cross-region moves). This removes the boundary effect: London's gross rate falls from 13.9% to 7.3% of population a year, close to the rest of England & Wales once their short moves are also dropped (5.4%).
- **Across regions, more migration still goes with more churn, but the link is moderate and largely London plus Scotland/NI.** r = 0.64 across all regions (0.80 with all cross-region moves), 0.49 excluding London, 0.30 within England & Wales outside London. Stable across windows (before the ONS method change, before COVID). Excluding only London ↔ London moves gives the same picture.
- **Net migration** is negative where churn is high (London loses residents to the rest of the UK), but this disappears once density is controlled for.
- **Over time the two diverge.** UK job reallocation fell between 2012–14 and 2017–19 (29.6% → 28.2%), while longer-distance migration **rose** (4.8% → 5.5% of population; all cross-region moves 7.0% → 8.0%). Within regions, years of higher migration do not go with more churn (fixed-effects panel). So, unlike the US "declining fluidity" story, falling UK job churn is **not** mirrored by falling geographic mobility, at least since 2012.
- **Caveats:**
  - **Short window:** migration data start in 2012 (they miss the steep fall in churn in the 2000s).
  - **Method change:** ONS changed its method from 2023.
  - **GP-registration data** capture young adults' moves with a lag.
  - **ITL1 boundaries are a rough proxy for distance:** a move across a regional border can still be short (e.g. Cheshire → Merseyside is excluded, but Cheshire → Staffordshire is not).
  - **Age:** migration is concentrated among young adults, so it overlaps with the age analysis.